In [0]:
df_billings=spark.read.table("workspace.`ds-datasets`.billings")
df_renewal=spark.read.table("workspace.`ds-raw-datasets`.renewal_call_cleaned")
df_call=spark.read.table("workspace.`ds-raw-datasets`.cc_calls_cleaned_final")

In [0]:
display(df_billings)

In [0]:
df_renewal.display()


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_billings=df_billings\
    .filter(F.col("prospect_outcome") != "Open") \
    .withColumn("datediff", F.datediff("closed_date", "prospect_renewal_date")) \
    .withColumn("index", F.monotonically_increasing_id())

# Use df_renewal instead of spark.table("post_renewal_churn.raw.renewal_calls")
df_calls = df_renewal \
    .filter(F.col("Analysed_Call") == "1") \

# ── 3. Join on co_ref — explicit condition to avoid ambiguous reference ──
df_joined = df_billings.join(
    df_calls,
    on=df_billings["Co_Ref"] == df_calls["Co_Ref"],
    how="left"
).filter(
    (F.col("Call_Date") >= F.col("prospect_renewal_date")) &
    (F.col("Call_Date") <= F.col("closed_date"))
).drop(df_calls["Co_Ref"], df_calls["Call_Date"])

# ── 4. Keep only the LATEST call per billing row (index) ────────────────
window = Window.partitionBy("index").orderBy(F.desc("Call_Date"))

df_latest_call = df_joined \
    .withColumn("rn", F.row_number().over(window)) \
    .filter(F.col("rn") == 1) \
    .drop("rn")

df_latest_call.display()
df_final = df_latest_call


In [0]:
df_final.count()

In [0]:
df_final.groupBy("prospect_outcome").count().display()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_billings=df_final\
    .filter(F.col("prospect_outcome") != "Open") \
    .withColumn("datediff", F.datediff("closed_date", "prospect_renewal_date")) \
    .withColumn("index", F.monotonically_increasing_id())

# Use df_renewal instead of spark.table("post_renewal_churn.raw.renewal_calls")
df_calls = df_call \
    .filter(F.col("Analysed_Call") == "1") \

# ── 3. Join on co_ref — explicit condition to avoid ambiguous reference ──
df_joined = df_billings.join(
    df_calls,
    on=df_billings["Co_Ref"] == df_calls["Co_Ref"],
    how="left"
).filter(
    (df_calls["Call_Date"] >= F.col("prospect_renewal_date"))
    
).drop(df_calls["Co_Ref"], df_calls["Call_Date"], df_calls["Analysed_Call"])

# ── 4. Keep only the LATEST call per billing row (index) ────────────────
window = Window.partitionBy("index").orderBy(F.desc("Call_Date"))

df_latest_call = df_joined \
    .withColumn("rn", F.row_number().over(window)) \
    .filter(F.col("rn") == 1) \
    .drop("rn")

df_latest_call.display()
df_final = df_latest_call
df_final.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.`ds-datasets`.final_joined_table")

In [0]:
df_final.toPandas().shape

In [0]:
df_final.groupBy("prospect_outcome").count().display()